In [ ]:
import polars as pl
import pyprojroot

ROOT = pyprojroot.here()

In [ ]:
from flaml.automl.automl import AutoML

# ---------------------------------------------------------
# 1. Carga y exclusión de columnas
# ---------------------------------------------------------
df_train_full = pl.read_parquet(ROOT/"data"/"processed"/"temporada1_limpio.parquet")

columnas_excluir = ['pitch_id', 'batter', 'pitcher', 'description', 'swing']
columnas_contaminadas = [
    'sz_top', 'sz_bot', 'sz_mid', 'strike_zone_size', 'relative_height', 'pitch_location',
    'is_strike_zone', 'is_shadow_zone', 'distance_to_corner'
]
todas_excluidas = columnas_excluir + columnas_contaminadas

feature_cols = [c for c in df_train_full.columns if c not in todas_excluidas]
print(f"Features usadas ({len(feature_cols)}):", feature_cols)


In [ ]:
# ---------------------------------------------------------
# 3. FLAML AutoML — solo LightGBM, logloss
# ---------------------------------------------------------
X = df_train_full.select(feature_cols).to_pandas()
y = df_train_full.select('swing').to_pandas()['swing']

automl = AutoML()

automl_settings = {
    "time_budget": 600,          # ajustar según lo que tengas disponible
    "metric": "log_loss",
    "task": "classification",
    "estimator_list": ["lgbm"],
    "eval_method": "cv",          # CV interna de FLAML, evita data snooping manual
    "n_splits": 5,
    "early_stop": True,
    "seed": 42,
}

automl.fit(X_train=X, y_train=y, **automl_settings)

print("Mejor modelo:", automl.model.estimator)
print("Mejores hiperparámetros:", automl.best_config)
print("Mejor logloss (CV):", automl.best_loss)

In [ ]:
# ---------------------------------------------------------
# 4. Generar predicciones para Kaggle (temporada 2, sin label)
# ---------------------------------------------------------
temporada2 = pl.read_parquet(ROOT/"data"/"processed"/"temporada2_limpio.parquet").to_pandas()

# Guardamos los IDs para el archivo final (Kaggle siempre pide el ID)
pitch_ids_t2 = temporada2['pitch_id']

# Filtramos X_test para que tenga EXACTAMENTE las mismas columnas que usamos para entrenar
X_test = temporada2.drop(columns=[col for col in todas_excluidas if col in temporada2.columns])

# Predecimos las probabilidades
pred_test = automl.predict_proba(X_test)[:, 1]

# Armamos el DataFrame final
predicciones = pl.DataFrame({
    'pitch_id': pitch_ids_t2,
    'swing_probability': pred_test
})

In [ ]:
 # Guardamos el archivo
ruta_prediccion = ROOT / "data" / "processed" / "predicciones.parquet"
predicciones.write_parquet(ruta_prediccion)